In [1]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 15.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system ==

In [2]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 109.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [3]:
!pip install dataloader

  Preparing metadata (setup.py) ... done
  Created wheel for dataloader: filename=dataloader-2.0-py3-none-any.whl size=10083 sha256=9a7a610dea00cd84d67aefd0117f3dcb992601c2268ff131ab868d0c74a6fe46
  Stored in directory: /root/.cache/pip/wheels/c7/b4/28/bf711e4f9bf69d9fc21b1c017d9b63eb13c3a6f35308088a0b
Successfully built dataloader


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!wandb login --relogin
#key = 3114d04ef3f8187e6f6852dd28ede0fa5a2ec32c

wandb: WARNING Using legacy-service, which is deprecated. If this is unintentional, you can fix it by ensuring you do not call `wandb.require('legacy-service')` and do not set the WANDB_X_REQUIRE_LEGACY_SERVICE environment variable.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [1]:
# 📦 Imports

import os
import gc
import re
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import wandb
import json
from datasets import Dataset as HFDataset
import torch.nn.functional as F

# Memory Optimization Setup
print("Setting up memory optimizations...")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
gc.collect()

# Enable gradient checkpointing and mixed precision
torch.backends.cudnn.benchmark = False  # Reduce memory fragmentation

Setting up memory optimizations...


In [2]:
def load_model_optimized(model_path):
    """Load model with memory optimizations"""
    print(f"Loading model from {model_path}...")

    # Try different loading strategies
    loading_configs = [
        # Strategy 1: Pure bfloat16
        {
            "torch_dtype": torch.bfloat16,
            "device_map": "auto",
        },
        # Strategy 2: Stage loading with CPU offload
        {
            "device_map": "auto",
            "torch_dtype": torch.bfloat16,
            "offload_folder": "offload_folder",
            "offload_state_dict": True,
        },
        # Strategy 3: Minimal GPU memory
        {
            "device_map": "balanced_low_0",
            "torch_dtype": torch.bfloat16,
        }
    ]

    # Try loading strategies until one works
    model = None
    for i, config in enumerate(loading_configs):
        try:
            print(f"Trying loading strategy {i+1}...")
            model = AutoModelForCausalLM.from_pretrained(
                model_path,
                **config
            )
            print(f"Successfully loaded with strategy {i+1}")
            break
        except RuntimeError as e:
            print(f"Strategy {i+1} failed: {e}")
            torch.cuda.empty_cache()
            gc.collect()

    if model is None:
        raise RuntimeError("All loading strategies failed. Try a machine with more memory.")

    # Enable memory optimizations
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
        print("Gradient checkpointing enabled")

    return model

In [3]:
class MedicalQADataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_length=128):
        """Create dataset from medical QA pairs in JSON"""
        self.tokenizer = tokenizer
        self.max_length = max_length

        # Convert HF dataset to examples
        self.examples = []
        for item in hf_dataset:
            self.examples.append({
                "text": item["text"],
                "reference": item["reference"]
            })

        # Tokenize all examples
        print(f"Tokenizing {len(self.examples)} examples...")
        self.encodings = []

        # Process in small batches to avoid OOM
        batch_size = 16
        for i in range(0, len(self.examples), batch_size):
            batch = self.examples[i:i+batch_size]
            batch_texts = [item["text"] for item in batch]

            # Tokenize this batch
            batch_encodings = tokenizer(
                batch_texts,
                truncation=True,
                padding="max_length",
                max_length=max_length,
                return_tensors="pt"
            )

            # Add to encodings
            for j in range(len(batch)):
                self.encodings.append({
                    key: val[j] for key, val in batch_encodings.items()
                })

        print(f"Created dataset with {len(self.encodings)} examples (max length {max_length})")

        # Log sample to wandb
        wandb.log({
            "dataset_size": len(self.examples),
            "max_length": max_length,
            "sample_text": self.examples[0]["text"] if self.examples else ""
        })

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        item = {key: val.clone() for key, val in self.encodings[idx].items()}
        item["labels"] = item["input_ids"].clone()
        return item

# Load multiple JSON datasets and combine them
def load_combined_json_datasets(json_paths, tokenizer, max_length=128):
    """Load and combine multiple JSON datasets"""
    print(f"Loading and combining {len(json_paths)} datasets...")

    all_train_data = []
    dataset_stats = {}

    # Process each JSON file
    for i, json_path in enumerate(json_paths):
        print(f"Loading dataset {i+1}/{len(json_paths)}: {json_path}")

        try:
            # Load JSON data
            with open(json_path, "r") as f:
                qa_data = json.load(f)

            # Get dataset name from filename
            dataset_name = os.path.basename(json_path).replace('.json', '')

            # Extract QA pairs
            dataset_items = []
            for topic in qa_data:
                for question, answer in topic['question_answer_pair']:
                    prompt = f"Answer this question about {dataset_name}: "
                    dataset_items.append({
                        "text": prompt + question,
                        "reference": answer,
                        "source": dataset_name
                    })

            # Add to combined dataset
            all_train_data.extend(dataset_items)

            # Track stats
            dataset_stats[dataset_name] = len(dataset_items)
            print(f"  Added {len(dataset_items)} examples from {dataset_name}")

        except Exception as e:
            print(f"Error loading {json_path}: {e}")

    # Create HF Dataset and split
    print(f"Creating dataset with {len(all_train_data)} total examples")
    hf_dataset = HFDataset.from_list(all_train_data).train_test_split(test_size=0.1)

    # Create our PyTorch dataset
    train_dataset = MedicalQADataset(
        hf_dataset["train"],
        tokenizer,
        max_length=max_length
    )

    # Log dataset info to wandb
    sample_questions = [item["text"] for item in all_train_data[:5]]
    wandb.config.update({
        "num_qa_pairs": len(all_train_data),
        "num_train": len(hf_dataset["train"]),
        "num_test": len(hf_dataset["test"]),
        "dataset_stats": dataset_stats,
        "sample_questions": sample_questions
    })

    return train_dataset

In [4]:
def identify_router_parameters(model):
    """Find router parameters in MoE model"""
    router_params = []
    router_param_names = []

    # Simple regex patterns for routers
    router_patterns = [r".*router.*", r".*gate.*", r".*expert_choice.*"]
    compiled_patterns = [re.compile(pattern, re.IGNORECASE) for pattern in router_patterns]

    # Find router parameters
    for name, param in model.named_parameters():
        if any(pattern.match(name) for pattern in compiled_patterns):
            router_param_names.append(name)
            router_params.append(param)
            print(f"Found router: {name}")

    # Log to wandb
    wandb.log({
        "num_router_parameters": len(router_params),
        "total_router_elements": sum(p.numel() for p in router_params)
    })

    print(f"Found {len(router_params)} router parameters")
    return router_param_names, router_params

In [5]:
def train_router_minimal(model, train_dataset, num_epochs=1):
    """Train router with minimal memory usage"""
    # Identify router parameters
    router_param_names, router_params = identify_router_parameters(model)

    if not router_params:
        print("No router parameters found!")
        return model

    # Create dataloader with batch size 1
    train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)

    # Freeze all parameters
    for name, param in model.named_parameters():
        param.requires_grad = False

    # Only unfreeze router parameters
    for name in router_param_names:
        for n, p in model.named_parameters():
            if n == name:
                p.requires_grad = True
                break

    # Create optimizer
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=1e-5
    )

    from torch.optim.lr_scheduler import StepLR
    scheduler = StepLR(optimizer, step_size=30, gamma=0.5)


    # Get device
    device = next(model.parameters()).device

    # Track steps for wandb
    global_step = 0

    # Train
    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        total_loss = 0
        step_count = 0

        for step, batch in enumerate(train_dataloader):
            if step >= 100:
                print(f"Stopping at step {step} to avoid OOM")
                break
            # Clear cache
            torch.cuda.empty_cache()

            try:
                # Move to device
                batch = {k: v.to(device) for k, v in batch.items()}

                # Forward
                with torch.amp.autocast('cuda'):
                    outputs = model(**batch)
                    loss = outputs.loss
                # Backward and optimize
                loss.backward()
                optimizer.step()
                if step % 30 == 0 and step > 0:
                    scheduler.step()
                    wandb.log({"learning_rate": scheduler.get_last_lr()[0]})
                optimizer.zero_grad()

                # Track loss
                loss_val = loss.item()
                total_loss += loss_val
                step_count += 1
                global_step += 1

                # Log to wandb
                wandb.log({
                    "step_loss": loss_val,
                    "epoch": epoch,
                    "step": step,
                    "global_step": global_step,
                })

                # Report progress
                print(f"Step {step}, Loss: {loss_val:.4f}")

                # Cleanup
                del outputs, loss, batch

            except RuntimeError as e:
                if "CUDA out of memory" in str(e):
                    print(f"OOM at step {step}, trying CPU...")

                    # Free memory
                    torch.cuda.empty_cache()

                    # Process on CPU instead
                    model_device = next(model.parameters()).device
                    model.cpu()

                    cpu_batch = {k: v.cpu() for k, v in batch.items()}
                    outputs = model(**cpu_batch)
                    loss = outputs.loss
                    loss.backward()

                    optimizer.step()
                    optimizer.zero_grad()

                    # Track loss
                    loss_val = loss.item()
                    total_loss += loss_val
                    step_count += 1
                    global_step += 1

                    # Log to wandb
                    wandb.log({
                        "step_loss": loss_val,
                        "epoch": epoch,
                        "step": step,
                        "global_step": global_step,
                        "processed_on_cpu": True
                    })

                    model.to(model_device)
                    print(f"Step {step} processed on CPU, Loss: {loss_val:.4f}")

                    # Cleanup
                    del outputs, loss, cpu_batch
                else:
                    raise e

        # Log epoch stats
        if step_count > 0:
            avg_loss = total_loss / step_count
            wandb.log({
                "epoch_loss": avg_loss,
                "epoch": epoch
            })
            print(f"Epoch {epoch+1} completed. Average loss: {avg_loss:.4f}")

        if epoch > 0:  # Save after first epoch
            checkpoint_path = save_checkpoint(
                model,
                router_param_names,
                epoch,
                "/content/drive/MyDrive/medmoe/trained_moe/checkpoints"
            )
    print("Training completed")
    return model

In [6]:
def save_router_params(model, router_param_names, output_path):
    """Save only router parameters to save memory"""
    router_state = {}

    for name in router_param_names:
        for n, p in model.named_parameters():
            if n == name:
                router_state[name] = p.detach().cpu()
                break

    # Save to file
    torch.save(router_state, output_path)
    print(f"Router parameters saved to {output_path}")

    # Log to wandb as artifact
    artifact = wandb.Artifact(
        "router_parameters",
        type="model",
        description="Trained MoE router parameters"
    )
    artifact.add_file(output_path)
    wandb.log_artifact(artifact)

In [7]:
def save_checkpoint(model, router_param_names, epoch, output_dir):
    """Save checkpoint of router parameters"""
    router_state = {}

    for name in router_param_names:
        for n, p in model.named_parameters():
            if n == name:
                router_state[name] = p.detach().cpu()
                break

    # Save to file
    checkpoint_path = f"{output_dir}/router_checkpoint_epoch_{epoch}.pt"
    torch.save(router_state, checkpoint_path)
    print(f"Checkpoint saved to {checkpoint_path}")

    # Log to wandb
    wandb.log({"checkpoint_saved": checkpoint_path, "epoch": epoch})

    return checkpoint_path

In [8]:
def load_model_with_router(
    model,
    router_params,
    torch_dtype=torch.bfloat16
):
    """Load model with trained router parameters"""

    # Update model with router parameters
    updated = 0
    for name, param in router_params.items():
        if name in model.state_dict():
            model.state_dict()[name].copy_(param)
            updated += 1

    print(f"Updated {updated}/{len(router_params)} router parameters")

    return model

In [9]:

wandb.init(
    project="moe-router-training",
    name="moe-router-llama3-ultramedical-20B",
    resume="allow",
    config={
        "learning_rate": 1e-4,
        "epochs": 5,
        "batch_size": 4,
        "max_length": 128,
        "platform": "colab"
    }
)

try:
    # Load model
    print("Loading model...")
    model_path = "/content/drive/MyDrive/medmoe/MoE/Llama3-UltraMedical-MoE-20B"

    try:
        # First try loading with minimal memory
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
        loading_strategy = "auto"
    except RuntimeError as e:
        print(f"First loading attempt failed: {e}")

        # Try with more aggressive offloading
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.bfloat16,
            device_map="balanced_low_0"
        )
        loading_strategy = "balanced_low_0"

    wandb.config.update({"loading_strategy": loading_strategy})

    # Enable memory optimizations
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
        wandb.config.update({"gradient_checkpointing": True})

    # Log system info
    wandb.config.update({
        "gpu_memory_gb": torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0,
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
        "pytorch_version": torch.__version__
    })

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    json_paths = [
        "/content/drive/MyDrive/medmoe/blood_heart_circulation_qa.json",
        "/content/drive/MyDrive/medmoe/bones_joints_muscles_qa.json",
        "/content/drive/MyDrive/medmoe/mental_behavior_qa.json"
    ]

    # Load and combine datasets
    train_dataset = load_combined_json_datasets(json_paths, tokenizer, max_length=wandb.config.max_length)

    # Train router
    model = train_router_minimal(model, train_dataset, num_epochs=wandb.config.epochs)

    # Save router parameters
    router_param_names, router_params = identify_router_parameters(model)
    router_param_dict = dict(zip(router_param_names, router_params))
    save_router_params(
        model,
        router_param_names,
        "/content/drive/MyDrive/medmoe/trained_moe/llama3_ultramedical_moe_router_params.pt"
    )



    model = load_model_with_router(model, router_param_dict)

    output_model_path = '/content/drive/MyDrive/medmoe/trained_moe/Llama3-UltraMedical-MoE-20B-with-Gate'

    print('Model saving to local folder ...')
    model.save_pretrained(output_model_path)
    tokenizer.save_pretrained(output_model_path)

    print("Done!")

finally:
    # Finish wandb run
    wandb.finish()

wandb: Currently logged in as: xj2193 (med-moe) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loading model...


Loading checkpoint shards:   0%|          | 0/9 [00:00<?, ?it/s]

Some weights of the model checkpoint at /content/drive/MyDrive/medmoe/MoE/Llama3-UltraMedical-MoE-20B were not used when initializing LlamaForCausalLM: ['model.layers.0.mlp.experts.0.down_proj.weight', 'model.layers.0.mlp.experts.0.gate_proj.weight', 'model.layers.0.mlp.experts.0.up_proj.weight', 'model.layers.0.mlp.experts.1.down_proj.weight', 'model.layers.0.mlp.experts.1.gate_proj.weight', 'model.layers.0.mlp.experts.1.up_proj.weight', 'model.layers.0.mlp.router.0.bias', 'model.layers.0.mlp.router.0.weight', 'model.layers.0.mlp.router.2.bias', 'model.layers.0.mlp.router.2.weight', 'model.layers.0.mlp.shared_experts.0.down_proj.weight', 'model.layers.0.mlp.shared_experts.0.gate_proj.weight', 'model.layers.0.mlp.shared_experts.0.up_proj.weight', 'model.layers.1.mlp.experts.0.down_proj.weight', 'model.layers.1.mlp.experts.0.gate_proj.weight', 'model.layers.1.mlp.experts.0.up_proj.weight', 'model.layers.1.mlp.experts.1.down_proj.weight', 'model.layers.1.mlp.experts.1.gate_proj.weight', 

Loading and combining 3 datasets...
Loading dataset 1/3: /content/drive/MyDrive/medmoe/blood_heart_circulation_qa.json
  Added 309 examples from blood_heart_circulation_qa
Loading dataset 2/3: /content/drive/MyDrive/medmoe/bones_joints_muscles_qa.json
  Added 155 examples from bones_joints_muscles_qa
Loading dataset 3/3: /content/drive/MyDrive/medmoe/mental_behavior_qa.json
  Added 144 examples from mental_behavior_qa
Creating dataset with 608 total examples
Tokenizing 547 examples...
Created dataset with 547 examples (max length 128)
Found router: model.layers.0.mlp.gate_proj.weight
Found router: model.layers.1.mlp.gate_proj.weight
Found router: model.layers.2.mlp.gate_proj.weight
Found router: model.layers.3.mlp.gate_proj.weight
Found router: model.layers.4.mlp.gate_proj.weight
Found router: model.layers.5.mlp.gate_proj.weight
Found router: model.layers.6.mlp.gate_proj.weight
Found router: model.layers.7.mlp.gate_proj.weight
Found router: model.layers.8.mlp.gate_proj.weight
Found rou

dataset_size,▁
epoch,▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▅▅▅▅▅▆▆▆▆▆▆▆▆████████████
epoch_loss,█▂▁▁▁
global_step,▁▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇█████
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
max_length,▁
num_router_parameters,▁▁
step,▂▄▅▆▇█▃▄▅▆▇█▁▂▂▃▄▅▅▆▇▇▇██▂▃▃▄▄▆▇▁▃▃▄▆▇▇█
step_loss,██▆▄▃▁▃▃▂▂▁▁▁▁▂▁▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_router_elements,▁▁
checkpoint_saved,/content/drive/MyDri...
